# Classifier Agent — Code Guide

**Owner:** Nadi Kyaw
**Target file:** `agents/nadi_classifier.py`

> This notebook is documentation only — do not run it. It mirrors the actual agent code block by block and explains what each part does in plain English, so it's easier for the team (and future me) to understand this code inside out.

## What this agent does

The Classifier Agent is the second step in the pipeline. It receives `processed_data.csv` from Aurora's Processing Agent (headlines + prices + ground-truth `up`/`down`/`neutral` labels) and generates a standalone Python script, `classifier.py`, that uses FinBERT (`ProsusAI/finbert`) to predict a label for every held-out **test** headline. It then runs that generated script and writes `predictions_test.csv`.

Unlike a normal "call a function, get predictions" agent, this one is a **code-generation agent**: instead of returning predictions directly, it writes out a `.py` file that does the classifying, and hands that file — plus the predictions it produced — to Sabina's Evaluator Agent. Sabina reads the code to spot issues (e.g. a hardcoded threshold) but does not re-run it herself (see `docs/data_contracts.md`, Handoff 2).

On a retune loop — Jack sends back `retune_request.json` when accuracy is too low — the agent regenerates `classifier.py` with the suggested hyperparameters. If `CLASSIFIER_USE_OLLAMA=true` is set, it will also try asking a local LLM (Ollama) to rewrite the `classify()` function itself, responding to Sabina's `focus_labels`/`code_notes`. That LLM-written code only ever gets used if it survives a strict validation gauntlet (parses, defines `classify`, and actually runs on the mock data producing the right columns and valid labels) — otherwise the deterministic template stays.

### Note on usefulness

Keeping this notebook in sync with `agents/nadi_classifier.py` matters most for the LLM code-gen path — it's easy to lose track of which guardrail rejects what, so writing it out in plain English here is as much for me as for anyone else reading this code.

## 1. Imports

These are the external libraries and sibling modules the agent needs:

- **`ast`** — parses Python source into a syntax tree; used to sanity-check LLM-generated code without running it
- **`json`** — reads `retune_request.json` and builds the Ollama request body
- **`os` / `sys`** — standard Python tools for working with file paths
- **`subprocess`** — runs the generated `classifier.py` as its own process, and runs candidate LLM code in a sandboxed temp dir
- **`tempfile`** — gives the LLM-code validation step a throwaway directory to run candidates in
- **`urllib.request`** — talks to the local Ollama server over plain HTTP (no extra package needed)
- **`typing.TypedDict`** — imported for type hints (the real shared state type is `PipelineState`, imported below)
- **`LABELS` / `PREDICTION_COLUMNS`** — shared contract constants from `agents/contracts.py`
- **`PipelineState`** — the shared state object that passes information between agents (defined in `agents/state.py`)
- **`Agent`** — Jack's base class (defined in `agents/base.py`) that every agent in the team subclasses

The `try`/`except` around the `agents.*` imports lets this file run both as part of the package (`from agents.base import Agent`) and as a standalone script (`from base import Agent`) — same trick Aurora and Sabina use, since `uv run agents/nadi_classifier.py` executes it directly rather than as a package.

In [ ]:
import ast
import json
import os
import subprocess
import sys
import tempfile
import urllib.request
from typing import TypedDict

# Allow running as a package or direct script
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
try:
    from agents.base import Agent
    from agents.contracts import LABELS, PREDICTION_COLUMNS
    from agents.state import PipelineState
except ModuleNotFoundError:
    from base import Agent
    from contracts import LABELS, PREDICTION_COLUMNS
    from state import PipelineState

## 2. Module-level configuration

Constants set once, at import time:

- **`OUTPUT_DIR`** — default folder (`outputs/`) for generated files when the caller doesn't specify a path
- **`USE_OLLAMA`** — read from the `CLASSIFIER_USE_OLLAMA` env var. Off (`false`) by default, so the classifier is fully deterministic unless someone opts in. This mirrors how Sabina and Freddi use their LLMs: the model may write code, but rules decide whether that code is allowed to run.
- **`OLLAMA_URL` / `OLLAMA_MODEL`** — where the local Ollama server lives and which model to ask
- **`OLLAMA_TIMEOUT_SECONDS`** / **`OLLAMA_MAX_TOKENS`** — code generation is slower than a one-line chat answer, so the timeout is longer (120s) and the output is capped (500 tokens) since `classify()` is a short function
- **`EXPECTED_PREDICTION_COLUMNS`** — an alias for the shared `PREDICTION_COLUMNS` contract (Handoff 2 in `docs/data_contracts.md`). This is the checklist used to validate LLM-written code before trusting it.
- **`MOCK_DATA`** — absolute path to `mock_data/processed_data.csv`, used as the sandbox input when test-running a candidate classifier

In [ ]:
OUTPUT_DIR = "outputs"

# --- Optional "agentic" code generation via a local LLM (Ollama) ---------------
# By default the classifier is generated from a fixed template (fully
# deterministic). When CLASSIFIER_USE_OLLAMA=true, we also ask a local LLM to
# ADAPT the classify() logic to the evaluator's feedback — but only keep its
# version if it passes a strict validation gauntlet, otherwise we fall back to
# the template. This mirrors how Sabina and Freddi use the LLM: the model may
# write code, but rules decide whether that code is allowed to run.
USE_OLLAMA = os.getenv("CLASSIFIER_USE_OLLAMA", "false").lower() == "true"
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.getenv("CLASSIFIER_OLLAMA_MODEL", "llama3.2")
OLLAMA_TIMEOUT_SECONDS = 120   # code generation is slower than a one-line answer
OLLAMA_MAX_TOKENS = 500        # a classify() function is short; cap it to bound time

# Handoff 2 columns live in agents/contracts.py; Nadi aliases them here because
# the validation code reads like a local checklist.
EXPECTED_PREDICTION_COLUMNS = PREDICTION_COLUMNS
MOCK_DATA = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))),
                         "mock_data", "processed_data.csv")

## 3. `CLASSIFIER_TEMPLATE` — the code we generate

This is the biggest piece of the file: a Python source template (as a string, with `{placeholders}`) for the actual `classifier.py` that gets written to disk and handed to Sabina. `generate_code()` (section 8) fills in the placeholders and writes this out.

Reading it top to bottom, the *generated* script does:

1. **Model loading** — if `MODEL_DIR` points at a folder that exists (a fine-tuned model from `agents/finetune_finbert.py`), load that; its head already outputs `up`/`down`/`neutral` directly. Otherwise fall back to pretrained `ProsusAI/finbert`, which only knows sentiment (`positive`/`negative`/`neutral`) — `SENTIMENT_TO_LABEL` translates that into a move direction.
2. **`classify(title)`** — tokenizes one headline, runs it through the model, softmaxes the logits into per-label probabilities, optionally *boosts* the probability of any label in `FOCUS_LABELS` by `BOOST_FACTOR` (this is how a retune can tell the classifier "pay more attention to `down`"), renormalizes so probabilities still sum to 1, then picks the top label — but only commits to it if its probability clears `THRESHOLD`; otherwise it falls back to `neutral`.
3. **`main(src, dst)`** — reads `processed_data.csv`, keeps only `split == "test"` rows (train rows were used to fit the fine-tuned model, so scoring them would leak), classifies each one, and writes `predictions_test.csv` with the contract columns from Handoff 2.

Because this whole block is a plain string, editing indentation or braces inside it is easy to get wrong — every literal `{` or `}` that should appear in the *generated* code (e.g. dict literals) is escaped as `{{`/`}}` so Python's `.format()` doesn't try to substitute it.

In [ ]:
CLASSIFIER_TEMPLATE = """
import csv
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "ProsusAI/finbert"
MODEL_DIR = {model_dir}   # folder with our fine-tuned weights, or None to use pretrained FinBERT
MAX_LENGTH = {max_length}
THRESHOLD = {threshold}
FOCUS_LABELS = {focus_labels}
BOOST_FACTOR = {boost_factor}
SENTIMENT_TO_LABEL = {{"positive": "up", "negative": "down", "neutral": "neutral"}}

# Authenticate to the HF Hub when a token is in the env (higher rate limits,
# faster downloads); fall back to unauthenticated access when it is absent.
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

if MODEL_DIR and os.path.isdir(MODEL_DIR):
    # We have a fine-tuned model. Its head already outputs up/down/neutral, so
    # the label mapping comes straight from the model config (no sentiment step).
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    ID2OURS = {{i: model.config.id2label[i].lower() for i in model.config.id2label}}
    print("[classifier] using fine-tuned model:", MODEL_DIR)
else:
    # No fine-tuned model: use pretrained FinBERT and translate its sentiment
    # (positive/negative/neutral) into a move direction (up/down/neutral).
    tokenizer = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, token=HF_TOKEN)
    ID2OURS = {{i: SENTIMENT_TO_LABEL[model.config.id2label[i].lower()] for i in model.config.id2label}}
    print("[classifier] using pretrained FinBERT (no fine-tuned model found)")
model.eval()

def classify(title: str) -> dict:
    inputs = tokenizer(title, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]
    
    by_label = {{ID2OURS[i]: float(p) for i, p in enumerate(probs)}}
    
    # Apply class boost to focus labels if specified
    for fl in FOCUS_LABELS:
        if fl in by_label:
            by_label[fl] *= BOOST_FACTOR
            
    # Normalize probabilities after boosting
    total_prob = sum(by_label.values())
    if total_prob > 0:
        by_label = {{k: round(v / total_prob, 4) for k, v in by_label.items()}}
        
    top_label = max(by_label, key=by_label.get)
    top_prob = by_label[top_label]
    
    predicted = top_label if top_prob >= THRESHOLD else "neutral"
    
    return {{
        "predicted_label": predicted,
        "confidence": top_prob,
        "prob_up": by_label["up"],
        "prob_down": by_label["down"],
        "prob_neutral": by_label["neutral"],
    }}

def main(src: str, dst: str) -> None:
    if not os.path.exists(src):
        raise FileNotFoundError(f"Source file not found: {{src}}")
        
    with open(src, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        
    if not rows:
        raise ValueError("Source file is empty")
        
    # Predict only the held-out test rows; the train rows were used to fit the
    # model, so scoring them would leak. No split column -> treat all as test.
    if "split" in rows[0]:
        rows = [r for r in rows if r["split"] == "test"]
        if not rows:
            raise ValueError("No split=test rows in input")

    out_cols = [c for c in rows[0].keys() if c != "split"] + [
        "predicted_label", "confidence", "prob_up", "prob_down", "prob_neutral", "split"
    ]
    
    # Predict each test row (rows already filtered to split=="test" above).
    for row in rows:
        pred_data = classify(row["article_title"])
        row.update(pred_data)
        row["split"] = "test"
        
    os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
    with open(dst, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=out_cols)
        writer.writeheader()
        writer.writerows(rows)

if __name__ == "__main__":
    src_file = sys.argv[1] if len(sys.argv) > 1 else "processed_data.csv"
    dst_file = sys.argv[2] if len(sys.argv) > 2 else "predictions_test.csv"
    main(src_file, dst_file)
"""

## 4. Helper — Calling Ollama

`_ollama_generate` sends a prompt to the local Ollama server and returns its text answer. Same simple `urllib` approach Sabina's evaluator uses elsewhere in the pipeline — no API key, no extra dependency, just a POST request to `http://localhost:11434/api/generate`.

`keep_alive: "10m"` keeps the model loaded in memory between calls (avoids a slow reload every retune), and `temperature: 0.2` keeps the rewritten `classify()` function close to deterministic rather than creative.

In [ ]:
def _ollama_generate(prompt: str) -> str:
    """Call the local Ollama server and return its text answer. Same simple
    urllib approach Sabina's evaluator uses — no API key, no extra package."""
    body = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "10m",
        "options": {"temperature": 0.2, "num_predict": OLLAMA_MAX_TOKENS},
    }).encode("utf-8")
    request = urllib.request.Request(
        OLLAMA_URL, data=body,
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urllib.request.urlopen(request, timeout=OLLAMA_TIMEOUT_SECONDS) as response:
        answer = json.loads(response.read().decode("utf-8"))
    return answer.get("response", "")

## 5. Helper — Building the LLM prompt

`_build_llm_prompt` turns Sabina's feedback (`focus_labels`, `code_notes`) into instructions for the LLM. It deliberately over-constrains the ask:

- Only rewrite `classify(title)` — nothing else in the file
- Must return a dict with exactly the five keys the template's `classify()` returns
- `predicted_label` must be one of the three valid labels
- Lists exactly which already-defined globals (`tokenizer`, `model`, `torch`, `ID2OURS`, `THRESHOLD`, `MAX_LENGTH`, `FOCUS_LABELS`, `BOOST_FACTOR`) it's allowed to reuse from the surrounding template
- No imports, no comments outside the function, no markdown fences

Being this explicit is what makes the validation gauntlet in section 6 tractable — the LLM is told the exact shape of a valid answer, so a well-behaved model's output should pass on the first try, and a malformed one gets caught rather than silently corrupting `classifier.py`.

In [ ]:
def _build_llm_prompt(focus_labels, code_notes) -> str:
    """Ask the LLM to rewrite ONLY the classify() function, adapting to the
    evaluator's feedback. We tell it exactly which globals it may use and what
    the function must return, so the output plugs straight into the template."""
    return (
        "You are the classifier agent in a stock-move prediction pipeline.\n"
        "Rewrite ONLY the Python function `classify(title)` to improve accuracy,\n"
        "responding to this feedback from the evaluator.\n\n"
        f"Weakest classes to focus on: {focus_labels}\n"
        f"Evaluator notes: {code_notes or 'none'}\n\n"
        "Rules:\n"
        "- Return ONLY the function, starting with `def classify(title):`.\n"
        "- It must return a dict with exactly these keys: predicted_label,\n"
        "  confidence, prob_up, prob_down, prob_neutral.\n"
        "- predicted_label must be one of: up, down, neutral.\n"
        "- You may use these already-defined globals: tokenizer, model, torch,\n"
        "  ID2OURS, THRESHOLD, MAX_LENGTH, FOCUS_LABELS, BOOST_FACTOR.\n"
        "- No imports, no comments outside the function, no markdown."
    )

## 6. Helpers — Splicing the LLM's answer into the template

Two small string-surgery helpers:

- **`_extract_code`** — LLMs often wrap code in ```` ```python ... ``` ```` fences even when told not to; this strips them off so what's left is just the function body.
- **`_swap_classify`** — takes the fully-formatted template code and replaces everything between `def classify(` and `def main(` with the LLM's new function, leaving the imports, model-loading block, and `main()` untouched. This is what lets us trust the *surrounding* code (it's still our deterministic template) while only the classification logic itself comes from the LLM.

In [ ]:
def _extract_code(text: str) -> str:
    """Strip ```python fences if the LLM wrapped its answer in them."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.startswith("python"):
            text = text[len("python"):]
    return text.strip()


def _swap_classify(default_code: str, new_classify: str) -> str:
    """Replace the classify() function in the generated code with the LLM's
    version, keeping everything else (imports, model loading, main()) intact."""
    start = default_code.index("def classify(")
    end = default_code.index("def main(", start)
    return default_code[:start] + new_classify.strip() + "\n\n\n" + default_code[end:]

## 7. The guardrail gauntlet

Before any LLM-rewritten `classify()` is allowed to replace the template version, it has to clear two checks, cheapest first:

- **`_looks_like_valid_classify`** — a *static* check, no model execution involved: does the whole file even parse as Python, and does it define a function literally named `classify`? This catches syntax errors and empty/garbled answers instantly.
- **`_runs_on_mock`** — the *real* guardrail: actually run the candidate script, as a subprocess, against `mock_data/processed_data.csv`, in a throwaway temp directory. It only passes if the run succeeds, produces `predictions_test.csv` with **exactly** the shared `PREDICTION_COLUMNS` order, every `predicted_label` is one of the shared `LABELS`, and every `confidence` parses as a float. Any exception, timeout, wrong columns, or bad value means reject.

Running the candidate as a subprocess (rather than `exec`-ing it in-process) means a broken or malicious candidate can't crash or corrupt the agent process itself — worst case, the subprocess fails and we fall back to the template.

In [ ]:
def _looks_like_valid_classify(code: str) -> bool:
    """Cheap static checks (no model needed): the whole file parses, and it
    defines a function literally named `classify`."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return False
    return any(isinstance(node, ast.FunctionDef) and node.name == "classify"
               for node in ast.walk(tree))


def _runs_on_mock(code: str) -> bool:
    """The strong guardrail: actually run the candidate classifier on the mock
    data and confirm it produces the exact contract columns AND sane values.
    Any error, wrong columns, bad label, or non-numeric confidence means we
    reject the LLM's code."""
    import csv as _csv

    with tempfile.TemporaryDirectory() as tmp:
        script = os.path.join(tmp, "candidate.py")
        out_csv = os.path.join(tmp, "out.csv")
        with open(script, "w", encoding="utf-8") as f:
            f.write(code)
        try:
            subprocess.run([sys.executable, script, MOCK_DATA, out_csv],
                           check=True, timeout=OLLAMA_TIMEOUT_SECONDS * 4,
                           capture_output=True)
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
            return False
        if not os.path.exists(out_csv):
            return False

        with open(out_csv, newline="", encoding="utf-8") as f:
            reader = _csv.DictReader(f)
            if reader.fieldnames != EXPECTED_PREDICTION_COLUMNS:
                return False
            rows = list(reader)

    if not rows:
        return False
    for row in rows:
        if row["predicted_label"] not in LABELS:
            return False
        try:
            float(row["confidence"])
        except (TypeError, ValueError):
            return False
    return True

## 8. `try_llm_classifier` — orchestrating the LLM attempt

This ties sections 4–7 together into one call: build the prompt from the retune request, call the LLM (or a fake `llm_fn` injected for tests), splice its answer into the template, and run it through both gauntlet checks. It returns the adapted code only if everything passes — otherwise `None`, and the caller (`generate_code`, next section) just keeps the deterministic template.

Note the `llm_fn` parameter: tests pass a fake function here instead of hitting a real Ollama server, and the network call itself is wrapped in a `try`/`except` so a down or unreachable Ollama server degrades to "keep the template" rather than crashing the whole classifier run.

In [ ]:
def try_llm_classifier(default_code, retune_req, llm_fn=None):
    """Try to get an LLM-adapted classifier. Returns the adapted code if it
    passes every guardrail, otherwise None (caller falls back to the template).

    llm_fn lets tests inject a fake LLM; in normal use it is the real Ollama call."""
    call_llm = llm_fn or _ollama_generate
    focus_labels = retune_req.get("focus_labels", [])
    code_notes = retune_req.get("code_notes") or retune_req.get("reason")

    try:
        answer = call_llm(_build_llm_prompt(focus_labels, code_notes))
    except Exception as error:  # network down, timeout, bad server, ...
        print(f"[nadi] LLM call failed ({error}); keeping the template classifier")
        return None

    new_classify = _extract_code(answer)
    candidate = _swap_classify(default_code, new_classify)

    # Gauntlet: cheap static check first, then the real run-on-mock check.
    if not _looks_like_valid_classify(candidate):
        print("[nadi] LLM code did not parse / had no classify(); using template")
        return None
    if not _runs_on_mock(candidate):
        print("[nadi] LLM code failed the mock run / contract check; using template")
        return None
    return candidate

## 9. `generate_code` — the main LangGraph node (writes `classifier.py`)

This is the first of the two nodes LangGraph runs for this agent. It receives the shared pipeline **state** and does the following:

1. **Read hyperparameters** — starts from defaults (`threshold=0.5`, `max_length=128`, `boost_factor=1.25`, no focus labels), then overrides them from `state["retune_request"]["suggested_params"]` if Jack sent one back.
2. **Resolve `model_dir`** — only used if `state["model_dir"]` is set *and* that folder actually exists on disk. This is opt-in on purpose: a stray folder left over from a previous run should never silently swap the model out from under a clean run.
3. **Fill in the template** — formats `CLASSIFIER_TEMPLATE` (section 3) with the resolved values.
4. **Optional LLM adaptation** — only attempted when there *is* a retune request AND either a fake `llm_fn` was injected (tests) or `USE_OLLAMA` is on. If `try_llm_classifier` (section 8) returns adapted code, that replaces the template version.
5. **Write `classifier.py`** — the single contract file Sabina reads.
6. **Archive to `classifier_history/`** — because `classifier.py` gets overwritten on every retune, each iteration's code is also saved as `classifier_history/classifier_iter{N}.py` (`N` = `0` for the first pass, then the retune's `iteration` number) so past attempts aren't lost. Audit trail only — nothing downstream reads it (see `docs/data_contracts.md`, Handoff 2).
7. **Return metadata** — `classifier_code_path`, `classifier_history_path`, and a `classifier_metadata` dict (model name + the hyperparameters used) that later ends up in the pipeline state for Sabina/Jack to inspect.

In [ ]:
def generate_code(state: PipelineState) -> dict:
    """LangGraph node to read parameters from state/retune request and write classifier.py."""
    threshold = 0.5
    max_length = 128
    focus_labels = []
    boost_factor = 1.25

    # Read from retune request if it exists in state
    retune_req = state.get("retune_request")
    if retune_req:
        suggested = retune_req.get("suggested_params", {})
        threshold = suggested.get("threshold", threshold)
        max_length = suggested.get("max_length", max_length)
        boost_factor = suggested.get("boost_factor", boost_factor)
        focus_labels = retune_req.get("focus_labels", focus_labels)

    code_path = state.get("classifier_code_path") or os.path.join(OUTPUT_DIR, "classifier.py")
    os.makedirs(os.path.dirname(code_path) or ".", exist_ok=True)

    # Use fine-tuned weights only when the caller explicitly asks for them via
    # state["model_dir"] AND that folder exists. This is opt-in on purpose: a
    # stray folder should never silently swap the model out from under a run.
    model_dir = state.get("model_dir")
    if model_dir and not os.path.isdir(model_dir):
        model_dir = None

    formatted_code = CLASSIFIER_TEMPLATE.format(
        model_dir=repr(model_dir),
        threshold=threshold,
        max_length=max_length,
        focus_labels=repr(focus_labels),
        boost_factor=boost_factor
    )

    # Optional "agentic" step: on a retune, let the LLM adapt the classify()
    # logic to the evaluator's feedback. We only keep its version if it passes
    # validation; otherwise formatted_code stays the deterministic template.
    llm_fn = state.get("llm_fn")
    if retune_req and (llm_fn or USE_OLLAMA):
        adapted_code = try_llm_classifier(formatted_code, retune_req, llm_fn)
        if adapted_code:
            formatted_code = adapted_code
            print("[nadi] using LLM-adapted classifier (passed validation)")

    with open(code_path, "w", encoding="utf-8") as f:
        f.write(formatted_code)

    print(f"[nadi] Generated classifier code at: {code_path}")

    # Keep a per-iteration copy so past retune attempts aren't lost when
    # classifier.py (the single contract file Sabina reads) gets overwritten.
    # iteration 0 = first pass before any retune; N = the Nth retune's code.
    iteration = retune_req.get("iteration", 0) if retune_req else 0
    history_dir = os.path.join(os.path.dirname(code_path) or ".", "classifier_history")
    os.makedirs(history_dir, exist_ok=True)
    history_path = os.path.join(history_dir, f"classifier_iter{iteration}.py")
    with open(history_path, "w", encoding="utf-8") as f:
        f.write(formatted_code)

    metadata = {
        "model_name": model_dir or "ProsusAI/finbert",
        "fine_tuning_params": {
            "threshold": threshold,
            "max_length": max_length,
            "focus_labels": focus_labels,
            "boost_factor": boost_factor
        }
    }

    return {
        "classifier_code_path": code_path,
        "classifier_history_path": history_path,
        "classifier_metadata": metadata
    }

## 10. `run_classifier` — the second LangGraph node (runs `classifier.py`)

Once `classifier.py` exists on disk, this node just runs it: `subprocess.run([sys.executable, code_path, data_path, pred_path], check=True)`.

Running it as a real subprocess (rather than importing and calling it) matters because the generated file is meant to be a standalone script — exactly what Sabina receives and could run herself. `check=True` means if the generated script raises an exception, this node raises too rather than silently producing an empty or partial `predictions_test.csv`.

`data_path` defaults to `mock_data/processed_data.csv` if the caller didn't supply `processed_data_path`, which is what lets this agent be built and tested before Aurora's real output exists (Golden rule 3: build against `mock_data/` first).

In [ ]:
def run_classifier(state: PipelineState) -> dict:
    """LangGraph node to run the generated classifier.py on processed_data.csv."""
    code_path = state["classifier_code_path"]
    data_path = state.get("processed_data_path") or "mock_data/processed_data.csv"
    pred_path = state.get("predictions_path") or os.path.join(OUTPUT_DIR, "predictions_test.csv")

    os.makedirs(os.path.dirname(pred_path) or ".", exist_ok=True)

    print(f"[nadi] Running classifier: {code_path} on {data_path} -> {pred_path}")
    
    # Run generated file as a subprocess
    subprocess.run([sys.executable, code_path, data_path, pred_path], check=True)

    print(f"[nadi] Predictions saved to: {pred_path}")

    return {
        "predictions_path": pred_path
    }

## 11. `build_graph` — wiring the two nodes together

A simple two-step LangGraph graph, same shape as Aurora's but with two nodes instead of one: `START → generate_code → run_classifier → END`. `generate_code`'s return value (the state dict with `classifier_code_path` etc.) is merged into the shared state before `run_classifier` runs, which is how `run_classifier` knows which file to execute.

In [ ]:
def build_graph(checkpointer):
    from langgraph.graph import StateGraph, START, END

    builder = StateGraph(PipelineState)
    builder.add_node("generate_code", generate_code)
    builder.add_node("run_classifier", run_classifier)
    builder.add_edge(START, "generate_code")
    builder.add_edge("generate_code", "run_classifier")
    builder.add_edge("run_classifier", END)
    return builder.compile(checkpointer=checkpointer)

## 12. `ClassifierAgent` class

Same team convention every agent follows (see Aurora's guide, section 6, and `agents/base.py`): subclass Jack's `Agent` base class, implement `build_graph` (just delegates to the module-level `build_graph` function above) and `run`.

`run` takes the contract file paths as named arguments — `processed_data`, `classifier_code`, `predictions`, plus the optional `retune_request` and `model_dir` — converts them to absolute paths, loads `retune_request.json` into the state if a path was given and the file exists (warning rather than crashing if it fails to parse — a bad retune file shouldn't take down the whole classifier), and calls `self._invoke()` from the base class to actually run the graph.

In [ ]:
class ClassifierAgent(Agent):
    """Classifier Agent (Nadi) behind the shared `.run()` interface."""

    def __init__(self, *, checkpointer=None, thread_id="classifier"):
        super().__init__(checkpointer=checkpointer, thread_id=thread_id)

    def build_graph(self, checkpointer):
        return build_graph(checkpointer)

    def run(self, processed_data: str, classifier_code: str, predictions: str,
            retune_request: str | None = None, model_dir: str | None = None) -> dict:
        """Runs the classifier generation and prediction step.

        Args:
            processed_data: Path to input processed_data.csv
            classifier_code: Output path for classifier.py
            predictions: Output path for predictions_test.csv
            retune_request: Optional path to input retune_request.json
            model_dir: Optional path to fine-tuned weights (outputs/finbert_finetuned).
                       When set and the folder exists, the classifier loads it instead
                       of pretrained FinBERT.
        """
        state = {
            "processed_data_path": os.path.abspath(processed_data),
            "classifier_code_path": os.path.abspath(classifier_code),
            "predictions_path": os.path.abspath(predictions),
        }
        if model_dir:
            state["model_dir"] = os.path.abspath(model_dir)
        if retune_request is not None and os.path.exists(retune_request):
            try:
                with open(retune_request, "r", encoding="utf-8") as f:
                    state["retune_request"] = json.load(f)
            except Exception as e:
                print(f"[nadi] Warning: Failed to parse retune_request file: {e}")
            
        return self._invoke(state)

## 13. Command-line entry point (TESTING)

Only runs when the script is called directly, e.g.:

```bash
uv run agents/nadi_classifier.py
uv run agents/nadi_classifier.py --retune-request outputs/retune_request.json
uv run agents/nadi_classifier.py --model-dir outputs/finbert_finetuned
```

Five optional arguments, all defaulting to the mock-data / `outputs/` paths so this runs end to end with no other agent's real output required (Golden rule 3):

- `--processed-data` — input CSV (default `mock_data/processed_data.csv`)
- `--classifier-code` — where to write the generated `classifier.py` (default `outputs/classifier.py`)
- `--predictions` — where to write `predictions_test.csv` (default `outputs/predictions_test.csv`)
- `--retune-request` — optional path to a `retune_request.json` to simulate a retune loop
- `--model-dir` — optional fine-tuned weights folder (from `agents/finetune_finbert.py`); only used if it exists

It builds a `ClassifierAgent()` directly rather than constructing the graph by hand, so testing from the terminal exercises the exact same code path Jack's Manager Agent will use.

In [ ]:
if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Run the Classifier Agent (Nadi).")
    parser.add_argument("--processed-data", default="mock_data/processed_data.csv", help="Input CSV path")
    parser.add_argument("--classifier-code", default="outputs/classifier.py", help="Output Python script path")
    parser.add_argument("--predictions", default="outputs/predictions_test.csv", help="Output predictions path")
    parser.add_argument("--retune-request", default=None, help="Input retune request JSON path")
    parser.add_argument("--model-dir", default=None,
                        help="Optional fine-tuned weights folder (e.g. outputs/finbert_finetuned)")

    args = parser.parse_args()

    agent = ClassifierAgent()
    res = agent.run(
        processed_data=args.processed_data,
        classifier_code=args.classifier_code,
        predictions=args.predictions,
        retune_request=args.retune_request,
        model_dir=args.model_dir,
    )
    print("\nClassifier completed. Predictions at:", res["predictions_path"])